# Bài 5: LLM hoạt động như thế nào

**Thời gian:** ~45 phút | **Chi phí:** $0 (không gọi API)

Trước khi bắt tay vào xây dựng AI agent, chúng ta cần hiểu bộ máy bên trong: **Mô hình ngôn ngữ lớn** (LLM — Large Language Model).

## LLM là gì?

Hãy tưởng tượng LLM như một **sinh viên đã đọc hết toàn bộ internet** — hàng tỷ trang web, sách, bài báo và mã nguồn. Sau khi đọc xong, sinh viên này học được các *quy luật* trong ngôn ngữ: câu văn được cấu trúc ra sao, các ý tưởng liên kết với nhau thế nào, từ ngữ nào thường đi sau từ ngữ nào.

Khi bạn hỏi LLM một câu hỏi, nó không tra cứu đáp án trong cơ sở dữ liệu. Thay vào đó, nó **dự đoán đoạn văn bản có khả năng cao nhất** nên xuất hiện tiếp theo, dựa trên tất cả những quy luật đã học được.

```
[Đầu vào của bạn] → [LLM] → [Văn bản được dự đoán]
```

**LLM** là viết tắt của **Large Language Model** (Mô hình ngôn ngữ lớn). Những model bạn có lẽ đã nghe tên — **Claude**, **GPT**, **Grok** — đều là LLM. Chúng khác nhau về dữ liệu huấn luyện, kiến trúc và khả năng, nhưng đều hoạt động trên cùng một nguyên lý cốt lõi: **dự đoán phần văn bản tiếp theo**.

Trong bài học này, bạn sẽ tìm hiểu các khái niệm nền tảng đằng sau LLM — không cần API key, chỉ cần Python và sự tò mò.

## Token — Cách LLM đọc văn bản

LLM không đọc từ ngữ giống cách bạn đọc. Chúng chia văn bản thành các **token** — những mảnh nhỏ, trung bình bằng khoảng **¾ một từ**.

Ví dụ:
- `"Hello"` → 1 token
- `"unbelievable"` → có thể bị tách thành `"un"` + `"believ"` + `"able"` → 3 token
- `"SEO"` → 1 token (viết tắt phổ biến)

**Tại sao điều này quan trọng?** Vì mỗi lần gọi API đều tính tiền **theo số token**. Khi bạn gửi văn bản cho Claude và nhận phản hồi, bạn đang trả tiền cho số token đi vào (input) và số token đi ra (output).

Dưới đây là bảng ước tính nhanh:

| Loại văn bản | Số token ước tính |
|------|---------------|
| Một tweet (280 ký tự) | ~60–80 |
| Một bài blog (1.000 từ) | ~1.300 |
| Một bài viết SEO đầy đủ (2.500 từ) | ~3.300 |

**Quy tắc nhẩm:** 1 từ ≈ 1,33 token. Hoặc ngược lại: 1 token ≈ ¾ một từ.

In [ ]:
# Hãy ước tính số token!
sample_texts = [
    "SEO is important for website rankings",
    "On-page SEO optimization involves improving title tags, meta descriptions, header tags, and content quality to help search engines understand your pages better.",
    "A comprehensive guide to building backlinks for your website that covers guest posting, broken link building, and digital PR strategies for improving domain authority.",
]

for text in sample_texts:
    word_count = len(text.split())
    estimated_tokens = int(word_count * 4 / 3)  # ước tính: 1 từ ≈ 1,33 token
    print(f'Văn bản: "{text[:50]}..."')
    print(f"  Số từ: {word_count}, Token ước tính: {estimated_tokens}")
    print()

## Context Window — "Bàn làm việc" của LLM

Hình dung bàn làm việc của bạn ở công ty. Bàn càng rộng, bạn càng có thể trải nhiều tài liệu ra và xử lý chúng cùng lúc. Bàn chật hẹp? Bạn chỉ nhìn được một trang.

**Context window** của LLM chính là cái bàn đó. Nó là **số token tối đa** (input + output cộng lại) mà model có thể xử lý trong một cuộc hội thoại.

| Model | Context window |
|-------|---------------|
| Claude Sonnet | 200.000 token |
| GPT-4o | 128.000 token |
| Grok | 131.072 token |

200.000 token tương đương khoảng **150.000 từ** — bằng cả một cuốn tiểu thuyết dày!

**Tại sao điều này quan trọng với agent:** Khi agent viết một bài SEO, mọi thứ đều phải nằm gọn trên cái bàn đó:
- **Hướng dẫn** mà bạn đã cung cấp cho agent
- **Ghi chú nghiên cứu** từ quá trình tìm kiếm web
- **Bài viết** mà agent đang soạn

Nếu tổng số token vượt quá context window, model sẽ không thể xử lý được. Hãy cùng tính xem chúng ta còn bao nhiêu dung lượng.

In [ ]:
# Context window của Claude chứa được bao nhiêu?
context_window = 200_000  # token

article_tokens = 3_300        # một bài viết SEO
research_notes_tokens = 2_000  # ghi chú nghiên cứu cho một bài
instructions_tokens = 500      # hướng dẫn cho agent

single_task = research_notes_tokens + instructions_tokens + article_tokens
print(f"Một bài viết sử dụng ~{single_task:,} token")
print(f"Chiếm {single_task / context_window * 100:.1f}% context window")
print(f"Claude có thể chứa ~{context_window // single_task} tác vụ viết bài trong context window")
print()
print(f"Một báo cáo 50 trang (~65.000 token) chiếm {65_000 / context_window * 100:.0f}% context window")
print(f"Vẫn còn thừa chỗ cho hướng dẫn và output!")

## Dữ liệu huấn luyện và Knowledge Cutoff

LLM học từ một bộ dữ liệu khổng lồ gồm văn bản — trang web, sách, mã nguồn, bài báo — được thu thập đến một ngày cụ thể. Ngày đó gọi là **knowledge cutoff** (thời điểm cắt kiến thức).

Sau ngày đó, model hoàn toàn **mù thông tin**. Nó không biết gì đã xảy ra.

Ví dụ:
- Nếu dữ liệu huấn luyện của Claude kết thúc vào đầu năm 2025, và bạn hỏi *"Xu hướng SEO nổi bật cuối năm 2025 là gì?"* — nó không thể biết. Nó có thể đoán, hoặc tệ hơn, bịa ra câu trả lời.
- Hỏi về bản cập nhật thuật toán Google mới nhất tuần trước? Cũng vậy.

Đó chính là lý do chúng ta cung cấp **công cụ** cho agent — chẳng hạn tìm kiếm web qua DataForSEO. Agent có thể tìm kiếm trên internet để lấy **thông tin mới, cập nhật** thay vì phụ thuộc vào dữ liệu huấn luyện đã cũ.

> **Xem trước:** Trong Bài 09, bạn sẽ trang bị cho agent công cụ tìm kiếm web để giải quyết đúng vấn đề này.

## Next Token Prediction — Cách LLM thực sự tạo văn bản

Đây là cơ chế cốt lõi. Khi LLM tạo văn bản, nó thực hiện vòng lặp sau:

1. **Nhận** văn bản đầu vào (prompt của bạn)
2. **Dự đoán** token tiếp theo có khả năng cao nhất
3. **Nối** token đó vào văn bản
4. **Lặp lại** bước 2–3 cho đến khi phản hồi hoàn chỉnh

Giống như tính năng gợi ý từ trên điện thoại, nhưng tinh vi hơn rất nhiều. Điện thoại gợi ý từ tiếp theo; LLM dự đoán token tiếp theo dựa trên hàng tỷ quy luật đã học.

### Temperature — Núm điều chỉnh sáng tạo

**Temperature** kiểm soát cách model chọn token tiếp theo:

- **Temperature 0** — Luôn chọn token có xác suất cao nhất. Kết quả **ổn định, nhất quán, chính xác**. Phù hợp để trích xuất dữ liệu, tạo output có cấu trúc.
- **Temperature 1** — Cho các token khác cơ hội công bằng. Kết quả **đa dạng hơn, sáng tạo hơn, đôi khi bất ngờ**. Phù hợp cho viết sáng tạo, brainstorming.

Hiểu đơn giản, đó là một núm vặn:
```
Chính xác/An toàn ←——— Temperature ———→ Sáng tạo/Rủi ro
       0.0                                       1.0
```

**Liên hệ với agent:** Khi bạn viết `instructions` cho agent, bạn đang tác động vào *token nào* model có xu hướng dự đoán. Nói "hãy viết ngắn gọn" khiến các token ngắn, trực tiếp có xác suất cao hơn. Nói "hãy sáng tạo và dùng ẩn dụ" thì dịch chuyển xác suất về phía ngôn ngữ giàu hình ảnh hơn.

In [ ]:
# Minh họa đơn giản cách temperature ảnh hưởng đến việc chọn từ
# Đừng lo về chi tiết code — hãy tập trung vào KẾT QUẢ khi chạy.
# Ý chính: temperature thấp = dễ đoán, temperature cao = bất ngờ.

import random

next_word_options = {
    "SEO is": {
        "important": 0.40,
        "essential": 0.25,
        "critical": 0.15,
        "fundamental": 0.10,
        "overrated": 0.05,
        "magical": 0.03,
        "bananas": 0.02,
    }
}

prompt = "SEO is"
words = list(next_word_options[prompt].keys())
weights = list(next_word_options[prompt].values())

print('Prompt: "SEO is ___"\n')
print("Temperature thấp (tập trung hơn):")
for i in range(5):
    adjusted = [w ** 2 for w in weights]
    total = sum(adjusted)
    adjusted = [w / total for w in adjusted]
    choice = random.choices(words, weights=adjusted, k=1)[0]
    print(f"  -> SEO is {choice}")

print("\nTemperature cao (sáng tạo hơn):")
for i in range(5):
    adjusted = [w ** 0.5 for w in weights]
    total = sum(adjusted)
    adjusted = [w / total for w in adjusted]
    choice = random.choices(words, weights=adjusted, k=1)[0]
    print(f"  -> SEO is {choice}")

Chạy ô code phía trên vài lần — bạn sẽ nhận ra:
- **Temperature thấp** hầu như luôn chọn "important" và "essential" (các ứng viên hàng đầu)
- **Temperature cao** thỉnh thoảng cho ra "magical" hay thậm chí "bananas"

Đó chính là lý do temperature quan trọng với nội dung SEO: bạn muốn bài viết chính xác, nhất quán (temperature thấp), chứ không muốn những bất ngờ sáng tạo xuất hiện giữa phần thống kê.

## Những gì LLM không làm được

Đây có lẽ là **phần quan trọng nhất** trong bài học này. LLM rất ấn tượng, nhưng chúng có những hạn chế thực sự ảnh hưởng trực tiếp đến công việc của bạn:

### Hallucination (Ảo giác)
LLM có thể **vô cùng tự tin** trong khi **sai hoàn toàn**. Chúng có thể:
- Bịa ra số liệu thống kê không tồn tại
- Trích dẫn nghiên cứu chưa bao giờ được viết
- Đưa thông tin lỗi thời như thể đó là sự thật hiện tại

Điều này xảy ra vì model đang *dự đoán văn bản nghe hợp lý*, chứ không phải *tra cứu sự thật*.

### Không có kiến thức thời gian thực
Như đã đề cập — sau ngày knowledge cutoff, model hoàn toàn mù. Nó không thể tự truy cập internet (trừ khi bạn trang bị công cụ cho nó).

### Không có bộ nhớ giữa các cuộc hội thoại
Mỗi khi bạn bắt đầu cuộc hội thoại mới, LLM khởi động lại từ đầu. Nó không nhớ gì bạn đã nói hôm qua. (Đó là lý do hệ thống chat của chúng ta ở Bài 18 cần tự xây dựng cơ chế ghi nhớ.)

### Tính toán không đáng tin
LLM **không phải máy tính bỏ túi**. Chúng có thể mắc lỗi số học cơ bản, đặc biệt với các con số lớn. Đừng bao giờ tin LLM làm sổ sách kế toán cho bạn.

---

**Tại sao đội SEO cần quan tâm:** Một bài viết được tạo tự động có thể chứa số liệu bịa, trích dẫn không tồn tại, hoặc thông tin lỗi thời — và tất cả nghe đều *rất thuyết phục*. Đó chính là lý do pipeline của chúng ta đưa bài viết vào trạng thái **"review"** (chờ duyệt), chứ không phải "published" (đã xuất bản) — **con người duyệt bài là bắt buộc**, không phải tùy chọn.

## Tổng kết Bài 5

| Thuật ngữ | Ý nghĩa | Tại sao đội SEO cần quan tâm |
|------|---------|-------------------|
| **Token** | ~3/4 một từ, đơn vị xử lý của LLM | Chi phí API tính theo token |
| **Context window** | Kích thước tối đa input + output | Giới hạn lượng nghiên cứu cho mỗi bài viết |
| **Knowledge cutoff** | Thời điểm dữ liệu huấn luyện kết thúc | Lý do agent cần công cụ tìm kiếm web |
| **Temperature** | Núm điều chỉnh sáng tạo (0=tập trung, 1=sáng tạo) | Kiểm soát tính nhất quán của văn phong |
| **Hallucination** | Output tự tin nhưng sai | Bài viết cần con người kiểm chứng |
| **Next token prediction** | Cách LLM tạo văn bản | Hiểu điều này giúp viết prompt tốt hơn |

**Những gì bạn đã học:**
- LLM là cỗ máy nhận diện quy luật, **dự đoán token tiếp theo** — chúng không thực sự "biết" điều gì
- **Token** là đơn vị tiền tệ của LLM — mỗi lần gọi API đều tốn tiền theo số token
- **Context window** đặt giới hạn cho lượng thông tin agent có thể xử lý cùng lúc
- **Temperature** kiểm soát sự cân bằng giữa output ổn định và sáng tạo
- LLM có thể **bịa thông tin**, **không có kiến thức thời gian thực**, và **không có bộ nhớ** — nên con người duyệt bài là không thể bỏ qua

**Bài tiếp theo:** Chúng ta sẽ học cách viết prompt hiệu quả — kỹ năng tạo nên sự khác biệt giữa một agent vô dụng và một agent mạnh mẽ.

## Bài tập

Hai câu đầu không cần viết code — chỉ cần suy nghĩ:

**1.** Bạn muốn tạo một bài viết SEO về *"giày chạy bộ tốt nhất 2026"*. Hạn chế nào của LLM khiến chủ đề này rủi ro nếu không có công cụ hỗ trợ?

<details>
<summary>Nhấn để xem đáp án</summary>

**Knowledge cutoff** — dữ liệu huấn luyện của LLM nhiều khả năng không bao gồm các sản phẩm năm 2026. Nếu không có công cụ tìm kiếm web, nó có thể bịa ra những mẫu giày không hề tồn tại.

</details>

**2.** Bài viết được tạo tự động về *"10 thống kê SEO"* chứa câu: *"93% trải nghiệm trực tuyến bắt đầu từ một công cụ tìm kiếm."* Bạn có nên tin con số này không? Tại sao?

<details>
<summary>Nhấn để xem đáp án</summary>

**Không** — rủi ro hallucination. LLM có thể đã bịa ra hoặc nhớ sai con số này. Ngay cả nếu đó là số liệu thật, nó có thể đã lỗi thời. Luôn xác minh số liệu thống kê từ nguồn gốc trước khi xuất bản.

</details>

**3.** Ước tính chi phí token để tạo một bài viết 2.000 từ, nếu mô hình tính **$3 cho mỗi triệu token đầu vào** và **$15 cho mỗi triệu token đầu ra**.

Gợi ý: 2.000 từ ≈ 2.700 token (output). Giả sử ~3.000 token input cho hướng dẫn + dàn ý.

Viết phép tính của bạn trong ô code bên dưới!

In [ ]:
# Bài tập: Tính chi phí ước tính từ câu hỏi 3 ở trên
# Input: ~3.000 token với giá $3 mỗi triệu
# Output: ~2.700 token với giá $15 mỗi triệu

# Code của bạn ở đây:
